In [ ]:
from pathlib import Path
import re
import warnings
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


warnings.filterwarnings("ignore")


# ============================================================
# 0. Path configuration
# ============================================================

def find_project_root():
    """
    Locate the project root from either a Python script or a notebook.

    Expected project structure:
    Project_root/
    ├─ Code/
    │  └─ classifier/
    ├─ Data/
    │  └─ ASIGranite.xls
    └─ Result/
    """
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (path / "Code").exists() and (path / "Data").exists():
            return path

    if start_dir.name.lower() == "classifier":
        return start_dir.parents[1]

    if start_dir.name.lower() == "code":
        return start_dir.parent

    return start_dir


PROJECT_ROOT = find_project_root()

RESULT_BASE_DIR = PROJECT_ROOT / "Result"

DATA_ROOT = (
    RESULT_BASE_DIR
    / "01_Foldwise_Preprocessing_and_Ratio_Features"
)

STABILITY_SUMMARY_FILE = (
    RESULT_BASE_DIR
    / "05_Stable_Cluster_Champions_and_Novel_Ratio_Candidates"
    / "rho090_stable_champions_and_ratio_candidates_summary.xlsx"
)

OUT_DIR = (
    RESULT_BASE_DIR
    / "08_Fixed_FiveFold_Multimodel_Performance_Comparison"
    / "MLP"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TYPE_COL = "Type"

IMPUTATION_METHODS = ["global_mean", "knn"]
N_OUTER_FOLDS = 5
CLASS_ORDER = ["A", "S", "I"]

LABEL_TO_INT = {
    "A": 0,
    "S": 1,
    "I": 2
}

INT_TO_LABEL = {
    0: "A",
    1: "S",
    2: "I"
}

TOP_N_STABLE_RATIOS = 20

MODEL_NAME = "MLP"


# ============================================================
# 1. MLP configuration
# ============================================================

SEED = 42

MLP_CONFIG = {
    "n_dim": 62,
    "num_classes": 3,
    "hidden": (256, 128, 64),
    "dropout": (0.25, 0.20, 0.15),
    "activation": "GELU",
    "batch_norm": True,
    "batch_size": 64,
    "max_epochs": 300,
    "patience": 20,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss_class_weighted",
    "scheduler": "ReduceLROnPlateau",
    "n_splits": 5,
    "random_state": SEED
}

VALIDATION_SIZE = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# 2. Utility functions
# ============================================================

def set_random_seed(seed):
    """
    Set random seeds for reproducibility.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def normalize_type_value(value):
    """
    Standardize A-type, S-type, and I-type labels to A, S, and I.
    """
    s = str(value).strip()

    if s in ["A", "A-type", "A-Type", "A_TYPE", "A type"]:
        return "A"

    if s in ["S", "S-type", "S-Type", "S_TYPE", "S type"]:
        return "S"

    if s in ["I", "I-type", "I-Type", "I_TYPE", "I type"]:
        return "I"

    return s


def display_feature_name(name):
    """
    Format feature names for display only.
    """
    s = str(name)

    s = s.replace("R_Major_", "")
    s = s.replace("R_Trace_", "")
    s = s.replace("A12O3", "Al2O3")
    s = s.replace("10000*Ga/Al", "10000xGa/Al")
    s = s.replace("10000*Ga/A1", "10000xGa/Al")
    s = s.replace("10000×Ga/Al", "10000xGa/Al")

    return s


def feature_key(name):
    """
    Build a normalized key for matching feature names across files.
    """
    s = str(name).strip()
    s = display_feature_name(s)
    s = s.replace("×", "*")
    s = s.replace("：", ":")
    s = re.sub(r"\s+", "", s)
    s = s.lower()

    return s


def resolve_one_feature(feature, columns):
    """
    Resolve one feature name against available dataframe columns.
    """
    columns = [str(c).strip() for c in columns]

    if feature in columns:
        return feature

    target_key = feature_key(feature)

    for col in columns:
        if feature_key(col) == target_key:
            return col

    fixed_feature_1 = str(feature).replace("A12O3", "Al2O3")
    fixed_feature_2 = str(feature).replace("Al2O3", "A12O3")

    for fixed_feature in [fixed_feature_1, fixed_feature_2]:
        if fixed_feature in columns:
            return fixed_feature

        fixed_key = feature_key(fixed_feature)

        for col in columns:
            if feature_key(col) == fixed_key:
                return col

    return None


def resolve_feature_list(features, columns):
    """
    Resolve a list of feature names against available dataframe columns.
    """
    resolved = []
    missing = []

    for feature in features:
        col = resolve_one_feature(feature, columns)

        if col is None:
            missing.append(feature)
        else:
            if col not in resolved:
                resolved.append(col)

    return resolved, missing


def get_classical_features(columns):
    """
    Get classical or commonly used granite-discrimination features
    if they are available in the current fold.
    """
    aliases = {
        "A/CNK": ["A/CNK", "ACNK"],
        "A/NK": ["A/NK", "ANK"],
        "10000xGa/Al": ["10000*Ga/Al", "10000xGa/Al", "10000×Ga/Al", "10000*Ga/A1"],
        "Zr+Nb+Ce+Y": ["Zr+Nb+Ce+Y", "Zr_Nb_Ce_Y"],
        "Sr/Y": ["Sr/Y", "R_Trace_Sr/Y"],
        "Rb/Sr": ["Rb/Sr", "R_Trace_Rb/Sr"],
        "Fe2O3t/MgO": ["Fe2O3t/MgO", "R_Major_Fe2O3t/MgO"],
    }

    selected = []

    for feature_name, options in aliases.items():
        found = None

        for feature in options:
            col = resolve_one_feature(feature, columns)

            if col is not None:
                found = col
                break

        if found is not None:
            selected.append(found)
        else:
            print(f"Warning: classical feature was not found: {feature_name}")

    return list(dict.fromkeys(selected))


def load_stable_novel_ratios():
    """
    Load stable candidate novel ratios from the Script 05 summary.
    """
    if not STABILITY_SUMMARY_FILE.exists():
        raise FileNotFoundError(
            f"Stability summary file was not found: {STABILITY_SUMMARY_FILE}"
        )

    xls = pd.ExcelFile(STABILITY_SUMMARY_FILE)

    if "stable_ratio_candidates" not in xls.sheet_names:
        raise ValueError(
            "The stability summary file does not contain the "
            "'stable_ratio_candidates' sheet."
        )

    df = pd.read_excel(
        STABILITY_SUMMARY_FILE,
        sheet_name="stable_ratio_candidates"
    )

    if "Feature" not in df.columns:
        raise ValueError(
            "The 'stable_ratio_candidates' sheet does not contain a Feature column."
        )

    if "Appearance_count" in df.columns:
        df = df[df["Appearance_count"] >= 6].copy()

    sort_cols = []
    ascending = []

    for col in [
        "Appearance_count",
        "Mean_champion_score",
        "Mean_SHAP_importance",
        "Mean_TopK_frequency"
    ]:
        if col in df.columns:
            sort_cols.append(col)
            ascending.append(False)

    if sort_cols:
        df = df.sort_values(sort_cols, ascending=ascending)

    stable_ratios = (
        df["Feature"]
        .dropna()
        .astype(str)
        .head(TOP_N_STABLE_RATIOS)
        .tolist()
    )

    return stable_ratios


def load_stable_feature_set():
    """
    Load the stable feature set from the Script 05 summary.

    Multiple sheet names are supported for compatibility with previous
    output versions.
    """
    if not STABILITY_SUMMARY_FILE.exists():
        return []

    xls = pd.ExcelFile(STABILITY_SUMMARY_FILE)

    candidate_sheets = [
        "stable_feature_candidates",
        "stable_features_for_paper",
        "core_stable_features",
        "champion_stability",
        "champion_stability_summary"
    ]

    for sheet_name in candidate_sheets:
        if sheet_name in xls.sheet_names:
            df = pd.read_excel(STABILITY_SUMMARY_FILE, sheet_name=sheet_name)

            if "Feature" in df.columns:
                return df["Feature"].dropna().astype(str).tolist()

    return []


def read_fold_data(method, fold):
    """
    Read one fixed outer-fold train/test dataset.
    """
    method_dir = DATA_ROOT / method

    train_path = method_dir / f"fold_{fold:02d}_train_with_ratios.xlsx"
    test_path = method_dir / f"fold_{fold:02d}_test_with_ratios.xlsx"

    if not train_path.exists():
        raise FileNotFoundError(f"Training-fold file was not found: {train_path}")

    if not test_path.exists():
        raise FileNotFoundError(f"Test-fold file was not found: {test_path}")

    train_df = pd.read_excel(train_path)
    test_df = pd.read_excel(test_path)

    train_df.columns = [str(c).strip() for c in train_df.columns]
    test_df.columns = [str(c).strip() for c in test_df.columns]

    if TYPE_COL not in train_df.columns:
        raise ValueError(f"Label column '{TYPE_COL}' was not found in: {train_path}")

    if TYPE_COL not in test_df.columns:
        raise ValueError(f"Label column '{TYPE_COL}' was not found in: {test_path}")

    train_df[TYPE_COL] = train_df[TYPE_COL].apply(normalize_type_value)
    test_df[TYPE_COL] = test_df[TYPE_COL].apply(normalize_type_value)

    return train_df, test_df


def build_feature_sets(train_columns):
    """
    Build the feature sets evaluated by this classifier.

    The statistical results from Script 07 are not used for feature selection.
    """
    stable_ratios = load_stable_novel_ratios()
    stable_features = load_stable_feature_set()

    classical = get_classical_features(train_columns)

    novel, novel_missing = resolve_feature_list(
        stable_ratios,
        train_columns
    )

    stable_set, stable_missing = resolve_feature_list(
        stable_features,
        train_columns
    )

    if novel_missing:
        print("Warning: the following stable novel ratios were not found in the data columns:")
        for feature in novel_missing[:20]:
            print("  ", feature)

    if stable_missing:
        print("Warning: the following stable feature-set features were not found in the data columns:")
        for feature in stable_missing[:20]:
            print("  ", feature)

    feature_sets = {
        "Classical_only": classical,
        "Stable_novel_ratios": novel,
        "Classical_plus_stable_novel": list(dict.fromkeys(classical + novel)),
        "Stable_feature_set": stable_set,
    }

    feature_sets = {
        name: features
        for name, features in feature_sets.items()
        if len(features) > 0
    }

    return feature_sets


def prepare_X_pair(train_df, test_df, features):
    """
    Prepare train/test feature matrices while keeping the same columns.

    Steps:
    1. Convert all selected features to numeric values.
    2. Replace inf with NaN.
    3. Remove columns that are entirely NaN in the training fold.
    4. Return float64 numpy arrays.
    """
    X_train_df = train_df[features].copy()
    X_test_df = test_df[features].copy()

    for col in features:
        X_train_df[col] = pd.to_numeric(X_train_df[col], errors="coerce")
        X_test_df[col] = pd.to_numeric(X_test_df[col], errors="coerce")

    X_train_df = X_train_df.replace([np.inf, -np.inf], np.nan)
    X_test_df = X_test_df.replace([np.inf, -np.inf], np.nan)

    all_nan_train_cols = X_train_df.columns[X_train_df.isna().all()].tolist()

    if all_nan_train_cols:
        print("Warning: the following features are entirely NaN in the training fold and were removed:")
        for col in all_nan_train_cols[:30]:
            print("  ", col)

        X_train_df = X_train_df.drop(columns=all_nan_train_cols)
        X_test_df = X_test_df.drop(columns=all_nan_train_cols)

    kept_features = X_train_df.columns.tolist()

    if len(kept_features) == 0:
        return None, None, []

    X_train = X_train_df.to_numpy(dtype=np.float64)
    X_test = X_test_df.to_numpy(dtype=np.float64)

    return X_train, X_test, kept_features


def encode_y_for_mlp(y_str):
    """
    Encode class labels for MLP training.

    A -> 0
    S -> 1
    I -> 2
    """
    y_series = pd.Series(y_str).astype(str)
    y_mapped = y_series.map(LABEL_TO_INT)

    if y_mapped.isna().any():
        bad_labels = y_series[y_mapped.isna()].unique()
        raise ValueError(f"Unrecognized class labels were found in y: {bad_labels}")

    return y_mapped.to_numpy(dtype=np.int64)


def decode_y_from_mlp(y_int):
    """
    Decode integer predictions back to A/S/I labels.
    """
    return np.array([INT_TO_LABEL[int(value)] for value in y_int])


# ============================================================
# 3. PyTorch MLP model
# ============================================================

class MLPNet(nn.Module):
    """
    Fully connected MLP with GELU activation, batch normalization, and dropout.
    """
    def __init__(
        self,
        n_dim,
        num_classes,
        hidden=(256, 128, 64),
        dropout=(0.25, 0.20, 0.15),
        batch_norm=True
    ):
        super().__init__()

        layers = []
        in_dim = n_dim

        for hidden_dim, drop_rate in zip(hidden, dropout):
            layers.append(nn.Linear(in_dim, hidden_dim))

            if batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))

            layers.append(nn.GELU())
            layers.append(nn.Dropout(drop_rate))

            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, num_classes))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def compute_class_weights(y_train):
    """
    Compute inverse-frequency class weights for CrossEntropyLoss.
    """
    counts = np.bincount(
        y_train,
        minlength=MLP_CONFIG["num_classes"]
    ).astype(float)

    counts[counts == 0] = 1.0

    weights = counts.sum() / (len(counts) * counts)

    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def preprocess_train_val_test(X_train_raw, y_train, X_test_raw, fold_seed):
    """
    Perform an internal stratified validation split and fit imputation/scaling
    only on the inner training subset.

    The outer test fold is only transformed.
    """
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=VALIDATION_SIZE,
        random_state=fold_seed
    )

    inner_train_idx, val_idx = next(splitter.split(X_train_raw, y_train))

    X_inner_raw = X_train_raw[inner_train_idx]
    y_inner = y_train[inner_train_idx]

    X_val_raw = X_train_raw[val_idx]
    y_val = y_train[val_idx]

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_inner = imputer.fit_transform(X_inner_raw)
    X_inner = scaler.fit_transform(X_inner)

    X_val = imputer.transform(X_val_raw)
    X_val = scaler.transform(X_val)

    X_test = imputer.transform(X_test_raw)
    X_test = scaler.transform(X_test)

    return X_inner, y_inner, X_val, y_val, X_test


def make_loader(X, y=None, batch_size=64, shuffle=False):
    """
    Build a PyTorch DataLoader.
    """
    X_tensor = torch.tensor(X, dtype=torch.float32)

    if y is None:
        dataset = TensorDataset(X_tensor)
    else:
        y_tensor = torch.tensor(y, dtype=torch.long)
        dataset = TensorDataset(X_tensor, y_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle
    )


def train_mlp_model(X_train_raw, y_train, X_test_raw, fold_seed):
    """
    Train the PyTorch MLP with early stopping.

    Returns the trained model and transformed outer test matrix.
    """
    set_random_seed(fold_seed)

    X_inner, y_inner, X_val, y_val, X_test = preprocess_train_val_test(
        X_train_raw,
        y_train,
        X_test_raw,
        fold_seed
    )

    n_dim = X_inner.shape[1]

    model = MLPNet(
        n_dim=n_dim,
        num_classes=MLP_CONFIG["num_classes"],
        hidden=MLP_CONFIG["hidden"],
        dropout=MLP_CONFIG["dropout"],
        batch_norm=MLP_CONFIG["batch_norm"]
    ).to(DEVICE)

    class_weights = compute_class_weights(y_inner)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=MLP_CONFIG["learning_rate"],
        weight_decay=MLP_CONFIG["weight_decay"]
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5
    )

    train_loader = make_loader(
        X_inner,
        y_inner,
        batch_size=MLP_CONFIG["batch_size"],
        shuffle=True
    )

    val_loader = make_loader(
        X_val,
        y_val,
        batch_size=MLP_CONFIG["batch_size"],
        shuffle=False
    )

    best_state = None
    best_val_loss = np.inf
    epochs_without_improvement = 0

    for epoch in range(1, MLP_CONFIG["max_epochs"] + 1):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                logits = model(xb)
                val_loss = criterion(logits, yb)

                val_losses.append(float(val_loss.item()))

        mean_val_loss = float(np.mean(val_losses))
        scheduler.step(mean_val_loss)

        if mean_val_loss < best_val_loss - 1e-6:
            best_val_loss = mean_val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= MLP_CONFIG["patience"]:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, X_test, best_val_loss, epoch


def predict_mlp(model, X_test):
    """
    Predict class labels using the trained MLP model.
    """
    model.eval()

    test_loader = make_loader(
        X_test,
        y=None,
        batch_size=MLP_CONFIG["batch_size"],
        shuffle=False
    )

    preds = []

    with torch.no_grad():
        for (xb,) in test_loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            pred = torch.argmax(logits, dim=1)
            preds.extend(pred.cpu().numpy().tolist())

    return np.array(preds, dtype=np.int64)


# ============================================================
# 4. Evaluation and plotting
# ============================================================

def evaluate_one(y_true, y_pred):
    """
    Calculate overall multiclass metrics.
    """
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return {
        "Accuracy": acc,
        "Balanced_accuracy": bacc,
        "Macro_precision": macro_precision,
        "Macro_recall": macro_recall,
        "Macro_F1": macro_f1,
        "Weighted_precision": weighted_precision,
        "Weighted_recall": weighted_recall,
        "Weighted_F1": weighted_f1,
    }


def plot_summary(summary_df):
    """
    Plot mean Macro-F1 values for all feature sets.
    """
    metric = "Macro_F1_mean"

    if summary_df.empty:
        print("summary_df is empty. No figure was generated.")
        return

    if metric not in summary_df.columns:
        print(f"{metric} was not found in summary_df. No figure was generated.")
        return

    plot_df = summary_df.sort_values(metric, ascending=True).copy()

    labels = (
        plot_df["Method"].astype(str)
        + " | "
        + plot_df["Feature_set"].astype(str)
    )

    plt.figure(figsize=(10, max(5, 0.35 * len(plot_df))), dpi=300)

    plt.barh(
        labels,
        plot_df[metric]
    )

    plt.xlabel("Mean Macro-F1", fontsize=12, fontweight="bold")
    plt.ylabel("Method | Feature set", fontsize=12, fontweight="bold")
    plt.title(
        f"{MODEL_NAME}: Fixed Five-Fold Performance",
        fontsize=14,
        fontweight="bold"
    )

    plt.tight_layout()

    out_png = OUT_DIR / f"{MODEL_NAME}_macroF1_summary.png"

    plt.savefig(out_png, bbox_inches="tight")
    plt.close()

    print("Figure saved to:", out_png)


# ============================================================
# 5. Main workflow
# ============================================================

def main():
    all_metric_rows = []
    all_classwise_rows = []
    all_cm_rows = []

    for method in IMPUTATION_METHODS:
        print(f"\n========== {MODEL_NAME} | {method} ==========")

        for fold in range(1, N_OUTER_FOLDS + 1):
            print(f"\nOuter fold {fold}")

            fold_seed = SEED + fold

            train_df, test_df = read_fold_data(method, fold)

            feature_sets = build_feature_sets(train_df.columns)

            for feature_set_name, features in feature_sets.items():
                print(f"\n  Feature set: {feature_set_name}")
                print(f"  Original n_features = {len(features)}")

                X_train_raw, X_test_raw, kept_features = prepare_X_pair(
                    train_df,
                    test_df,
                    features
                )

                if X_train_raw is None or X_test_raw is None or len(kept_features) == 0:
                    print(
                        f"Skipping {MODEL_NAME} | {method} | fold {fold} | "
                        f"{feature_set_name}: no usable numeric features."
                    )
                    continue

                y_train_str = train_df[TYPE_COL].astype(str).to_numpy()
                y_test = test_df[TYPE_COL].astype(str).to_numpy()

                y_train = encode_y_for_mlp(y_train_str)

                print("  Cleaned n_features =", len(kept_features))
                print("  X_train dtype:", X_train_raw.dtype)
                print("  X_test dtype:", X_test_raw.dtype)
                print("  y_train dtype:", y_train.dtype)
                print("  y_train classes:", sorted(np.unique(y_train)))
                print("  X_train shape:", X_train_raw.shape)
                print("  X_test shape:", X_test_raw.shape)
                print("  Device:", DEVICE)

                try:
                    model, X_test, best_val_loss, epochs_run = train_mlp_model(
                        X_train_raw,
                        y_train,
                        X_test_raw,
                        fold_seed
                    )

                    y_pred_int = predict_mlp(model, X_test)
                    y_pred = decode_y_from_mlp(y_pred_int)

                except Exception as exc:
                    print(
                        f"Model training or prediction failed: "
                        f"{MODEL_NAME} | {method} | fold {fold} | {feature_set_name}"
                    )
                    print("Error message:", exc)
                    continue

                metric_row = {
                    "Model": MODEL_NAME,
                    "Method": method,
                    "Outer_fold": fold,
                    "Feature_set": feature_set_name,
                    "N_features": len(kept_features),
                    "Features": "; ".join(kept_features),
                    "N_train": len(train_df),
                    "N_test": len(test_df),
                    "Best_validation_loss": best_val_loss,
                    "Epochs_run": epochs_run,
                }

                metric_row.update(
                    evaluate_one(y_test, y_pred)
                )

                all_metric_rows.append(metric_row)

                report = classification_report(
                    y_test,
                    y_pred,
                    labels=CLASS_ORDER,
                    output_dict=True,
                    zero_division=0
                )

                for cls in CLASS_ORDER:
                    all_classwise_rows.append({
                        "Model": MODEL_NAME,
                        "Method": method,
                        "Outer_fold": fold,
                        "Feature_set": feature_set_name,
                        "Class": cls,
                        "Precision": report[cls]["precision"],
                        "Recall": report[cls]["recall"],
                        "F1": report[cls]["f1-score"],
                        "Support": report[cls]["support"],
                    })

                cm = confusion_matrix(
                    y_test,
                    y_pred,
                    labels=CLASS_ORDER
                )

                for i, true_cls in enumerate(CLASS_ORDER):
                    for j, pred_cls in enumerate(CLASS_ORDER):
                        all_cm_rows.append({
                            "Model": MODEL_NAME,
                            "Method": method,
                            "Outer_fold": fold,
                            "Feature_set": feature_set_name,
                            "True_class": true_cls,
                            "Pred_class": pred_cls,
                            "Count": int(cm[i, j]),
                        })

                print(
                    f"  Completed: Macro-F1 = {metric_row['Macro_F1']:.4f}, "
                    f"Balanced accuracy = {metric_row['Balanced_accuracy']:.4f}, "
                    f"Accuracy = {metric_row['Accuracy']:.4f}, "
                    f"epochs = {epochs_run}"
                )

    metrics_df = pd.DataFrame(all_metric_rows)
    classwise_df = pd.DataFrame(all_classwise_rows)
    cm_df = pd.DataFrame(all_cm_rows)

    if metrics_df.empty:
        raise ValueError(
            "No model results were generated. Please check whether all training attempts failed."
        )

    metric_cols = [
        "Accuracy",
        "Balanced_accuracy",
        "Macro_precision",
        "Macro_recall",
        "Macro_F1",
        "Weighted_precision",
        "Weighted_recall",
        "Weighted_F1",
        "Best_validation_loss",
        "Epochs_run",
    ]

    group_cols = [
        "Model",
        "Method",
        "Feature_set",
        "N_features"
    ]

    summary_df = (
        metrics_df
        .groupby(group_cols)[metric_cols]
        .agg(["mean", "std"])
        .reset_index()
    )

    summary_df.columns = [
        col[0] if col[1] == "" else f"{col[0]}_{col[1]}"
        for col in summary_df.columns.to_flat_index()
    ]

    out_xlsx = OUT_DIR / f"{MODEL_NAME}_fixed5fold_results.xlsx"

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        metrics_df.to_excel(
            writer,
            sheet_name="fold_metrics",
            index=False
        )

        summary_df.to_excel(
            writer,
            sheet_name="summary_mean_std",
            index=False
        )

        classwise_df.to_excel(
            writer,
            sheet_name="classwise_metrics",
            index=False
        )

        cm_df.to_excel(
            writer,
            sheet_name="confusion_matrix_long",
            index=False
        )

        pd.DataFrame([MLP_CONFIG]).to_excel(
            writer,
            sheet_name="model_parameters",
            index=False
        )

    print("\nExcel file saved to:", out_xlsx)

    plot_summary(summary_df)

    print("\n========== Current model summary ==========")

    show_cols = [
        "Model",
        "Method",
        "Feature_set",
        "N_features",
        "Accuracy_mean",
        "Balanced_accuracy_mean",
        "Macro_precision_mean",
        "Macro_recall_mean",
        "Macro_F1_mean",
        "Weighted_F1_mean",
        "Epochs_run_mean"
    ]

    show_cols = [
        col for col in show_cols
        if col in summary_df.columns
    ]

    summary_to_show = (
        summary_df[show_cols]
        .sort_values(["Method", "Macro_F1_mean"], ascending=[True, False])
        .reset_index(drop=True)
    )

    print(summary_to_show.to_string(index=False))

    print("\n========== Best result for the current model ==========")

    best_row = (
        summary_df
        .sort_values("Macro_F1_mean", ascending=False)
        .head(1)
    )

    print(best_row[show_cols].to_string(index=False))

    summary_png = OUT_DIR / f"{MODEL_NAME}_macroF1_summary.png"

    print("\nCompleted:", MODEL_NAME)
    print("Result file:", out_xlsx)
    print("Figure file:", summary_png)
    print("Output directory:", OUT_DIR)


# ============================================================
# 6. Program entry point
# ============================================================

if __name__ == "__main__":
    main()